# 🧠 AFFERENT URL Threat Detection — Training Notebook v4

> **Dataset:** [URL-AF on Kaggle](https://www.kaggle.com/datasets/w5lasmbc/url-af)
> **Classes:** 4-way `benign` / `phishing` / `malware` / `other` (gambling/"judol" + misc local threats)
> **Status:** Evaluation of the v3-as-run notebook against production targets, with targeted fixes.

## Apa yang berubah dari v3-as-run (ringkasan evaluasi)

Notebook ini adalah revisi berbasis **bukti empiris** dari hasil run v3 sebelumnya (log terlampir), bukan tebakan ulang dari nol. Temuan kunci dari run v3 dan perbaikannya:

| # | Temuan di v3-as-run | Bukti dari log | Perbaikan di v4 |
|---|---|---|---|
| 1 | **Cascade (gate confidence Tier‑1) justru lebih buruk dari Stacker biasa** | Stacker Acc 0.8329 vs Cascade Acc 0.7664 (‑6.6pp), FPR 0.0543→0.0751 | Gate diganti ke **P(malicious) terkalibrasi**, threshold di-*tune* di validation set (bukan konstanta 0.85/0.15 tebakan), dibandingkan head-to-head dengan versi lama |
| 2 | Fitur "eksternal" (`domain_age`, `ssl_valid`, dst) di-hardcode **‑1 untuk semua baris** → tidak pernah membawa informasi, dan hard-override rule yang bergantung padanya **tidak pernah aktif** | Komentar di kode asli: *"Won't fire on offline test since domain_age stubs are ‑1"* | Kolom stub dihapus dari training set (memang **tidak bisa** direkonstruksi dari dataset historis statis — lihat catatan §2). Override rule didokumentasikan ulang sebagai **serving-time only** |
| 3 | CNN-only ablation **menyamai/mengalahkan** CNN-BiLSTM penuh, dan ~4× lebih cepat; LSTM-only paling lambat & paling lemah | CNN-only Acc 0.8268 vs CNN-BiLSTM 0.8203, ~10s/epoch vs ~44s/epoch | Tier‑2 default diganti **CNN + Self-Attention** (buang komponen rekuren yang terbukti jadi liability), BiLSTM/LSTM-only tetap disimpan sebagai pembanding ablasi |
| 4 | **OOD "other"/judol tidak pernah diprediksi** (0 dari 404 URL zero‑shot Indonesia), padahal ground truth 350/404 berlabel "other" | Predicted class distribution OOD: benign 294, phishing 110, **other 0** | Ditambahkan **leksikon judol Indonesia** sebagai fitur lexical eksplisit + fitur `brand_mismatch` untuk phishing |
| 5 | Phishing paling lemah di semua model (PR-AUC & FPR per-kelas terburuk) | Cascade per-class: phishing PR-AUC 0.6943, FPR 0.1604 (tertinggi) | Fitur `brand_mismatch` (brand di URL tapi bukan domain terdaftar) + kalibrasi tier lebih baik |
| 6 | Tidak ada pengecekan kebocoran domain train/test, tidak ada kalibrasi confidence, tidak ada benchmark latency nyata | — | Ditambahkan cek overlap domain, kalibrasi isotonic per tier, benchmark latency CPU nyata |

**Catatan jujur di muka:** target agregat v3 (F1≥95%, FPR<2%, dst) kemungkinan besar **tidak realistis** untuk dicapai murni dari sinyal lexical + karakter URL saja — itu memang batas informasi yang tersedia di dataset statis ini. Notebook ini akan mendorong performa semaksimal mungkin dalam batas itu, lalu memberi verdict jujur di §13 tentang seberapa dekat/jauh dari "layak produksi", plus rekomendasi konkret untuk menutup gap (data judol lebih banyak, pipeline fitur reputasi real-time di serving, dst).

---


---
## §0 — Setup, Data Loading & Validation

In [ ]:
!pip install -q tldextract optuna xgboost scikit-learn statsmodels

In [ ]:
import os, re, math, hashlib, warnings, time, json, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse

import tldextract, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder, label_binarize, MinMaxScaler, StandardScaler
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | XGBoost: {xgb.__version__}")

In [ ]:
# ── Kaggle / Local dataset path ───────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/w5lasmbc/url-af
candidates = [
    Path("/kaggle/input/url-af"),
    Path("/kaggle/input/url-af/processed"),
    Path("/kaggle/input/datasets/w5lasmbc/url-af"),   # observed path in v3-as-run log
    Path("../dataset/processed"),
    Path("dataset/processed"),
    Path("./"),
]
DATA_DIR = None
for c in candidates:
    if (c / "train.csv").exists():
        DATA_DIR = c
        break
if DATA_DIR is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("train.csv"))
    if found:
        DATA_DIR = found[0].parent
if DATA_DIR is None:
    DATA_DIR = Path("/kaggle/input/url-af")  # default Kaggle path

print(f"Detected DATA_DIR: {DATA_DIR}")

if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/afferent_models")
else:
    OUT_DIR = Path("./afferent_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {OUT_DIR}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
ood_df   = pd.read_csv(DATA_DIR / "indonesian_threat_casestudy.csv")

print(f"\nTrain : {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  OOD: {len(ood_df):,}")
print("Columns:", train_df.columns.tolist())
print("\nLabel dist (train):")
print(train_df["label"].value_counts())

LABEL_ORDER = ["benign", "phishing", "malware", "other"]
le = LabelEncoder()
le.fit(LABEL_ORDER)
for df in [train_df, val_df, test_df]:
    df["label_enc"] = le.transform(df["label"])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_).tolist())))

In [ ]:
# ── Integrity check (soft warning, not a hard fail — dataset may be re-versioned) ──
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest().upper()

for fname in ["train.csv", "val.csv", "test.csv", "indonesian_threat_casestudy.csv"]:
    p = DATA_DIR / fname
    if p.exists():
        print(f"  [OK]  {fname}  ({p.stat().st_size/1024:.1f} KB, sha256={sha256_file(p)[:16]}...)")
    else:
        print(f"  [MISSING] {fname}")

### Production-readiness gate #1 — data validation

Sebelum melatih apa pun, kita cek 3 hal yang sering diam-diam merusak evaluasi model deteksi URL:

1. **Konsistensi label**: apakah `is_malicious` selalu konsisten dengan `label != benign`?
2. **Kebocoran domain antar split**: dataset ini diklaim *"domain-grouped, zero-leakage split"* di deskripsinya — kita verifikasi klaim itu, bukan percaya begitu saja.
3. **Cakupan panjang URL**: apakah `MAX_LEN=200` karakter untuk model karakter-level cukup, atau banyak URL yang terpotong?

In [ ]:
# 1) Label vs is_malicious consistency
if "is_malicious" in train_df.columns:
    mismatch = ((train_df["label"] != "benign").astype(int) != train_df["is_malicious"].astype(int))
    print(f"Label/is_malicious mismatch (train): {mismatch.mean()*100:.2f}%  ({mismatch.sum():,} rows)")
    if mismatch.any():
        print(train_df.loc[mismatch, ["label", "is_malicious"]].value_counts().head(10))
else:
    print("Column 'is_malicious' not found in train.csv — skipping consistency check.")

# 2) Domain overlap across splits (leakage diagnostic)
DOMAIN_COL = "registered_domain" if "registered_domain" in train_df.columns else None
if DOMAIN_COL:
    train_domains = set(train_df[DOMAIN_COL].dropna().str.lower())
    for name, df in [("val", val_df), ("test", test_df)]:
        overlap = df[DOMAIN_COL].str.lower().isin(train_domains)
        print(f"  {name}: {overlap.mean()*100:.2f}% of rows share a registered_domain with train "
              f"({overlap.sum():,}/{len(df):,})")
    TEST_OVERLAP_MASK = test_df[DOMAIN_COL].str.lower().isin(train_domains).values
else:
    print("Column 'registered_domain' not found — cannot check domain leakage directly.")
    TEST_OVERLAP_MASK = np.zeros(len(test_df), dtype=bool)

# 3) URL length coverage vs planned MAX_LEN=200
lens = train_df["url"].astype(str).str.len()
print(f"\nURL length (train): mean={lens.mean():.1f}  p95={lens.quantile(.95):.0f}  "
      f"p99={lens.quantile(.99):.0f}  max={lens.max()}")
print(f"  Share of URLs longer than 200 chars (would be truncated): {(lens>200).mean()*100:.2f}%")

---
## §1 — EDA & Distribution Analysis

In [ ]:
def path_depth(url):
    try:
        parsed = urlparse(url if str(url).startswith("http") else "http://" + str(url))
        return len([p for p in parsed.path.split("/") if p])
    except Exception:
        return 0

for df in [train_df, val_df, test_df]:
    df["url_len"] = df["url"].astype(str).str.len()
    df["path_depth"] = df["url"].apply(path_depth)

print("Mean URL length by class:")
print(train_df.groupby("label")["url_len"].mean().round(1))
print("\nMean path depth by class:")
print(train_df.groupby("label")["path_depth"].mean().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (col, xlim) in zip(axes, [("url_len", 300), ("path_depth", 15)]):
    for label, color in zip(LABEL_ORDER, ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"]):
        data = train_df[train_df["label"] == label][col]
        ax.hist(data.clip(upper=xlim), bins=40, alpha=0.5, label=label, color=color, density=True)
    ax.set(title=f"{col} distribution by class", xlabel=col, ylabel="density")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "eda_url_structure.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
train_df["label"].value_counts().reindex(LABEL_ORDER).plot(
    kind="bar", ax=ax, color=["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"])
ax.set(title="Class distribution (train)", ylabel="count")
plt.tight_layout()
plt.savefig(OUT_DIR / "eda_class_distribution.png", dpi=150)
plt.show()

---
## §2 — Feature Engineering

**Catatan penting soal fitur "reputasi eksternal" (domain age, SSL, ASN, redirect chain):**
Fitur-fitur ini **tidak bisa direkonstruksi dari dataset historis statis seperti ini**. Umur domain
yang relevan adalah umur domain **pada saat URL itu pertama kali dilaporkan** — bukan umur domain
hari ini. Domain phishing dari kampanye 2019 kemungkinan besar sudah mati, di-parkir, atau
diregistrasi ulang oleh pihak lain sekarang, sehingga WHOIS lookup hari ini tidak merepresentasikan
kondisi historisnya. v3-as-run diam-diam menghindari masalah ini dengan meng-hardcode kolom-kolom
tsb sebagai `-1` konstan untuk **setiap baris** — yang secara efektif membuatnya tidak pernah
dipakai model manapun (baik pun rule hard-override yang bergantung padanya).

Konsekuensinya: fitur reputasi eksternal **bukan bagian dari training notebook ini** — itu memang
harus jadi fitur *serving-time* (dihitung live oleh backend saat URL baru masuk), bukan fitur
training. Lihat stub `apply_realtime_overrides()` di §13 untuk kontrak yang perlu diimplementasikan
tim backend.

Yang **bisa** ditambah di training (semua murni dari string URL, tanpa network call) — ini fokus
perbaikan utama di v4, terutama untuk menutup dua gap yang terlihat di evaluasi v3-as-run:
(1) deteksi phishing (brand impersonation) dan (2) deteksi judol/other pada data OOD Indonesia.

In [ ]:
BRANDS = [
    "paypal", "facebook", "google", "apple", "amazon", "microsoft",
    "netflix", "instagram", "twitter", "linkedin", "dropbox", "whatsapp",
    "outlook", "office365", "icloud", "ebay", "adobe",
    # Indonesian brands frequently impersonated in local phishing
    "klikbca", "mandiri", "bri", "bni", "permata", "cimb", "btn", "danamon",
    "tokopedia", "shopee", "gojek", "grab", "traveloka", "dana", "ovo",
    "gopay", "linkaja", "bpjs", "pajak", "dukcapil",
]

HIGH_RISK_TLDS = {
    ".xyz", ".top", ".club", ".info", ".online", ".site", ".live",
    ".icu", ".tk", ".ml", ".ga", ".cf", ".gq", ".pw", ".cc",
    ".buzz", ".fun", ".click", ".link", ".loan", ".work", ".win",
    ".cyou", ".sbs", ".cfd", ".rest", ".bond", ".monster", ".beauty",
    ".lol", ".mom", ".quest", ".lat", ".vip",
}

SUSPICIOUS_KW = [
    "login", "signin", "verify", "secure", "account", "update",
    "confirm", "banking", "wallet", "password", "credential",
    "webscr", "cmd=_", "click", "redirect", "suspend", "limited",
    "unlock", "validate",
    "daftar", "masuk", "konfirmasi", "verifikasi", "keamanan", "blokir",
]

# Indonesian online-gambling ("judol") lexicon — targeted fix for the OOD blind
# spot in v3-as-run (0/404 "other" predictions on the zero-shot judol set).
# Kept intentionally high-precision (distinctive slot/game-brand terms) rather
# than generic marketing words ("resmi", "terpercaya", "deposit") which would
# false-positive on ordinary Indonesian e-commerce/fintech sites.
JUDOL_KW = [
    "slot", "gacor", "maxwin", "rtp", "jackpot", "togel", "toto",
    "judi", "taruhan", "pragmatic", "olympus", "zeus", "mahjong",
    "scatter", "spaceman", "sbobet", "rungkad", "freebet", "bonanza",
    "starlight", "gatotkaca", "wildbandito",
]

SHORTENERS = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly",
    "adf.ly", "cutt.ly", "s.id", "bit.do", "shorte.st", "rebrand.ly", "tiny.cc",
}


def levenshtein(s1, s2):
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[-1] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]


def brand_min_distance(domain: str, brands=BRANDS) -> int:
    domain = domain.lower()
    best = 99
    for b in brands:
        if b in domain:
            return 0
        for i in range(max(1, len(domain) - len(b) + 1)):
            d = levenshtein(domain[i:i + len(b)], b)
            if d < best:
                best = d
        best = min(best, levenshtein(domain, b))
    return best


def brand_domain_mismatch(url_lower: str, domain: str, brands=BRANDS) -> int:
    """1 if a brand name appears in the URL but is NOT the registered domain —
    the classic 'paypal.com.evil.xyz' / 'secure-paypal.evil.com' pattern."""
    domain = domain.lower()
    for b in brands:
        if b in url_lower and b not in domain:
            return 1
    return 0


def shannon_entropy(s):
    if not s:
        return 0.0
    cnt = Counter(s)
    n = len(s)
    return -sum((v / n) * math.log2(v / n) for v in cnt.values())


def max_consecutive_consonants(s):
    vowels = set("aeiou")
    best = cur = 0
    for c in s.lower():
        if c.isalpha() and c not in vowels:
            cur += 1
            best = max(best, cur)
        elif c.isalpha():
            cur = 0
    return best


def extract_features(url: str) -> dict:
    url = str(url)
    try:
        parsed = urlparse(url if url.startswith("http") else "http://" + url)
    except Exception:
        parsed = urlparse("http://unknown")
    ext = tldextract.extract(url)
    domain = ext.domain or ""
    subdomain = ext.subdomain or ""
    suffix = ext.suffix or ""
    registered_domain = f"{domain}.{suffix}".strip(".")
    path = parsed.path or ""
    query = parsed.query or ""
    host = parsed.netloc.split(":")[0]
    path_parts = [p for p in path.split("/") if p]
    is_ip = bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))
    lev_min = brand_min_distance(domain)
    url_lower = url.lower()
    judol_hits = sum(1 for kw in JUDOL_KW if kw in url_lower)

    return {
        "url_len": len(url),
        "domain_len": len(host),
        "path_len": len(path),
        "query_len": len(query),
        "path_depth": len(path_parts),
        "query_param_cnt": len(query.split("&")) if query else 0,
        "num_digits": sum(c.isdigit() for c in url),
        "num_dashes": url.count("-"),
        "num_dots": url.count("."),
        "num_slashes": url.count("/"),
        "num_special": sum(c in "@?=&%+#" for c in url),
        "num_upper": sum(c.isupper() for c in url),
        "subdomain_count": len([s for s in subdomain.split(".") if s]),
        "has_at": int("@" in url),
        "has_port": int(":" in host),
        "uses_https": int(parsed.scheme == "https"),
        "is_ip": int(is_ip),
        "tld_risk": float(f".{suffix}".lower() in HIGH_RISK_TLDS),
        "levenshtein_min": lev_min,
        "keyword_count": sum(1 for kw in SUSPICIOUS_KW if kw in url_lower),
        "entropy_domain": shannon_entropy(domain),
        "entropy_url": shannon_entropy(url),
        "has_brand_name": int(any(b in url_lower for b in BRANDS)),
        "brand_mismatch": brand_domain_mismatch(url_lower, domain),
        "ratio_digits": sum(c.isdigit() for c in url) / max(len(url), 1),
        "ratio_dashes": url.count("-") / max(len(host), 1),
        "typosquat_subs": len(re.findall(r"[0-9](?=[a-z])|[a-z](?=[0-9])", domain)),
        # -- new in v4 --
        "has_punycode": int("xn--" in host.lower()),
        "is_shortener": int(registered_domain.lower() in SHORTENERS),
        "judol_score": judol_hits,
        "has_judol_kw": int(judol_hits > 0),
        "max_consonant_run": max_consecutive_consonants(domain),
        "vowel_ratio_domain": sum(c in "aeiou" for c in domain.lower()) / max(len(domain), 1),
        "www_prefix": int(subdomain.lower() == "www"),
        "num_www_like": int(bool(re.search(r"w{2,}\d?\.", host.lower()))),
        "ip_in_query": int(bool(re.search(r"\d{1,3}(?:\.\d{1,3}){3}", query))),
    }

In [ ]:
print("Extracting features for all splits...")
t0 = time.time()

def extract_df(df):
    return pd.DataFrame(df["url"].apply(extract_features).tolist(), index=df.index)

X_train_lex = extract_df(train_df); print(f"  train: {time.time()-t0:.1f}s")
X_val_lex   = extract_df(val_df);   print(f"  val:   {time.time()-t0:.1f}s")
X_test_lex  = extract_df(test_df);  print(f"  test:  {time.time()-t0:.1f}s")
X_ood_lex   = extract_df(ood_df);   print(f"  ood:   {time.time()-t0:.1f}s  [total]")

LEX_COLS = X_train_lex.columns.tolist()
print(f"\nEngineered features: {len(LEX_COLS)}")

y_train = train_df["label_enc"].values
y_val   = val_df["label_enc"].values
y_test  = test_df["label_enc"].values

In [ ]:
# Sanity check: does the new judol lexicon actually discriminate the "other" class?
print("Mean judol_score by class (train):")
print(X_train_lex.assign(label=train_df["label"].values).groupby("label")["judol_score"].mean().round(3))
print("\nMean brand_mismatch by class (train):")
print(X_train_lex.assign(label=train_df["label"].values).groupby("label")["brand_mismatch"].mean().round(3))

In [ ]:
# Feature-label correlation heatmap
corr = X_train_lex[LEX_COLS].copy()
corr["label"] = y_train
c = corr.corr()["label"].drop("label").sort_values()
fig, ax = plt.subplots(figsize=(6, 9))
c.plot(kind="barh", ax=ax, color=["#e74c3c" if v < 0 else "#2ecc71" for v in c.values])
ax.set(title="Feature correlation with label_enc", xlabel="Pearson r")
plt.tight_layout()
plt.savefig(OUT_DIR / "feature_correlations.png", dpi=150)
plt.show()

---
## §3 — Baseline ML Models (sanity checks)

In [ ]:
def evaluate(name, y_true, y_pred, y_proba=None):
    acc  = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average="weighted")
    f1_m = f1_score(y_true, y_pred, average="macro")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(len(le.classes_))))
    fp   = cm.sum(axis=0) - np.diag(cm)
    fn   = cm.sum(axis=1) - np.diag(cm)
    tn   = cm.sum() - (fp + fn + np.diag(cm))
    mean_fpr = (fp / (fp + tn + 1e-9)).mean()

    r = {"Model": name, "Accuracy": acc, "F1-weighted": f1_w,
         "F1-macro": f1_m, "Precision": prec, "Recall": rec, "FPR": mean_fpr}
    if y_proba is not None:
        y_bin = label_binarize(y_true, classes=list(range(len(le.classes_))))
        r["ROC-AUC"] = roc_auc_score(y_bin, y_proba, multi_class="ovr", average="macro")
        r["PR-AUC"]  = average_precision_score(y_bin, y_proba, average="macro")
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"  Acc: {acc:.4f}  F1-w: {f1_w:.4f}  F1-m: {f1_m:.4f}")
    print(f"  Prec: {prec:.4f}  Rec: {rec:.4f}  FPR: {mean_fpr:.4f}")
    if "PR-AUC" in r:
        print(f"  ROC-AUC: {r['ROC-AUC']:.4f}  PR-AUC: {r['PR-AUC']:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return r

RESULTS = []
X_tr = X_train_lex[LEX_COLS].fillna(-1).values
X_vl = X_val_lex[LEX_COLS].fillna(-1).values
X_te = X_test_lex[LEX_COLS].fillna(-1).values
print("Feature matrix shape:", X_tr.shape)

In [ ]:
# Logistic Regression
scaler_lr = StandardScaler()
X_tr_lr = scaler_lr.fit_transform(X_tr)
X_te_lr = scaler_lr.transform(X_te)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, n_jobs=-1)
lr.fit(X_tr_lr, y_train)
RESULTS.append(evaluate("Logistic Regression", y_test, lr.predict(X_te_lr), lr.predict_proba(X_te_lr)))

In [ ]:
# Naive Bayes
scaler_nb = MinMaxScaler()
nb = GaussianNB()
nb.fit(scaler_nb.fit_transform(X_tr), y_train)
RESULTS.append(evaluate("Naive Bayes", y_test, nb.predict(scaler_nb.transform(X_te)),
                        nb.predict_proba(scaler_nb.transform(X_te))))

In [ ]:
# SVM (LinearSVC + calibration for probabilities)
scaler_svm = StandardScaler()
X_tr_svm = scaler_svm.fit_transform(X_tr)
X_te_svm = scaler_svm.transform(X_te)

svm_base = LinearSVC(C=1.0, random_state=SEED, max_iter=5000)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_tr_svm, y_train)
RESULTS.append(evaluate("SVM (LinearSVC calibrated)", y_test, svm.predict(X_te_svm), svm.predict_proba(X_te_svm)))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=15, min_samples_leaf=5, random_state=SEED)
dt.fit(X_tr, y_train)
RESULTS.append(evaluate("Decision Tree", y_test, dt.predict(X_te), dt.predict_proba(X_te)))

---
## §4 — Ensemble Baselines (RF standalone, XGBoost standalone, Tier-1b TF-IDF)

In [ ]:
# Random Forest standalone (not part of the ensemble — kept only as a baseline reference)
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=3,
                             n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_train)
RESULTS.append(evaluate("Random Forest (standalone)", y_test, rf.predict(X_te), rf.predict_proba(X_te)))

In [ ]:
# XGBoost standalone (fixed hyperparameters, no tuning) — reference point before Optuna tuning
dtrain = xgb.DMatrix(X_tr, label=y_train)
dval   = xgb.DMatrix(X_vl, label=y_val)
dtest  = xgb.DMatrix(X_te, label=y_test)

xgb_params_base = {
    "objective": "multi:softprob", "num_class": len(le.classes_),
    "eval_metric": ["mlogloss", "merror"],
    "eta": 0.1, "max_depth": 6, "subsample": 0.8,
    "colsample_bytree": 0.8, "seed": SEED,
    "device": "cuda" if DEVICE == "cuda" else "cpu",
    "verbosity": 0,
}
xgb_base = xgb.train(xgb_params_base, dtrain, num_boost_round=300,
                     evals=[(dval, "val")], early_stopping_rounds=20, verbose_eval=50)
p_xgb_base = xgb_base.predict(dtest).reshape(-1, len(le.classes_))
RESULTS.append(evaluate("XGBoost Standalone (lexical-only)", y_test, p_xgb_base.argmax(1), p_xgb_base))

### Tier-1b — character n-gram TF-IDF + linear model

Sinyal yang ditangkap TF-IDF karakter n-gram tumpang tindih sebagian dengan fitur lexical
tangan-buatan di atas, tapi menangkap pola substring (mis. potongan brand yang di-typosquat,
token judol) yang tidak selalu tertangkap statistik ringkasan. Modelnya linear/sangat ringan
(sub-milidetik per URL), jadi praktis gratis untuk ditambahkan sebagai sinyal tambahan ke stacker.

In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                        max_features=4000, sublinear_tf=True)
Xtr_tfidf = tfidf.fit_transform(train_df["url"].astype(str))
Xvl_tfidf = tfidf.transform(val_df["url"].astype(str))
Xte_tfidf = tfidf.transform(test_df["url"].astype(str))
Xood_tfidf = tfidf.transform(ood_df["url"].astype(str))

tier1b = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=50,
                       random_state=SEED, n_jobs=-1, class_weight="balanced")
tier1b.fit(Xtr_tfidf, y_train)

p_t1b_val  = tier1b.predict_proba(Xvl_tfidf)
p_t1b_test = tier1b.predict_proba(Xte_tfidf)
p_t1b_ood  = tier1b.predict_proba(Xood_tfidf)
RESULTS.append(evaluate("Tier-1b TF-IDF+SGD (char n-grams)", y_test,
                        tier1b.predict(Xte_tfidf), p_t1b_test))

with open(OUT_DIR / "tier1b_tfidf.pkl", "wb") as f: pickle.dump(tfidf, f)
with open(OUT_DIR / "tier1b_sgd.pkl", "wb") as f: pickle.dump(tier1b, f)

---
## §5 — Tier 1: XGBoost (Optuna-tuned, maximize PR-AUC on val)

In [ ]:
def xgb_objective(trial):
    params = {
        "objective": "multi:softprob", "num_class": len(le.classes_), "eval_metric": "mlogloss",
        "eta":             trial.suggest_float("eta", 0.03, 0.3, log=True),
        "max_depth":       trial.suggest_int("max_depth", 4, 10),
        "min_child_weight":trial.suggest_int("min_child_weight", 1, 10),
        "subsample":       trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":           trial.suggest_float("gamma", 0.0, 5.0),
        "lambda":          trial.suggest_float("lambda", 0.1, 10.0, log=True),
        "alpha":           trial.suggest_float("alpha", 0.0, 5.0),
        "seed": SEED, "verbosity": 0,
        "device": "cuda" if DEVICE == "cuda" else "cpu",
    }
    m = xgb.train(params, dtrain, num_boost_round=500,
                  evals=[(dval, "val")], early_stopping_rounds=30, verbose_eval=False)
    p = m.predict(dval).reshape(-1, len(le.classes_))
    y_bin = label_binarize(y_val, classes=list(range(len(le.classes_))))
    return average_precision_score(y_bin, p, average="macro")

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f"Best PR-AUC (val): {study.best_value:.4f}")
print("Best params:", study.best_params)

In [ ]:
best = study.best_params
best.update({"objective": "multi:softprob", "num_class": len(le.classes_),
             "eval_metric": ["mlogloss", "merror"],
             "seed": SEED,
             "device": "cuda" if DEVICE == "cuda" else "cpu"})

tier1_xgb = xgb.train(best, dtrain, num_boost_round=800,
                      evals=[(dval, "val")], early_stopping_rounds=30, verbose_eval=50)
tier1_xgb.save_model(str(OUT_DIR / "tier1_xgboost.json"))
print("Tier-1 XGBoost saved.")

p_t1_val  = tier1_xgb.predict(dval).reshape(-1, len(le.classes_))
p_t1_test = tier1_xgb.predict(dtest).reshape(-1, len(le.classes_))
pred_t1   = p_t1_test.argmax(1)
res_t1    = evaluate("Tier-1 XGBoost (tuned + lexical, raw)", y_test, pred_t1, p_t1_test)
RESULTS.append(res_t1)

In [ ]:
# Feature importance
fi = pd.DataFrame({"feat": list(tier1_xgb.get_score(importance_type="gain").keys()),
                   "imp": list(tier1_xgb.get_score(importance_type="gain").values())})
fi = fi.sort_values("imp", ascending=True).tail(25)
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(fi["feat"], fi["imp"], color="#3498db")
ax.set(title="Tier-1 XGBoost — Feature Importance (gain)")
plt.tight_layout()
plt.savefig(OUT_DIR / "tier1_feature_importance.png", dpi=150)
plt.show()

### Kalibrasi Tier-1 — kenapa ini penting untuk cascade

Cascade v3-as-run men-*gate* berdasarkan `max(softmax)` XGBoost mentah. Tree ensemble terkenal
sering **overconfident**: bisa bilang "confidence 0.9" padahal salah jauh lebih sering dari 10%
di titik itu. Kita cek dulu, baru kalibrasi dengan isotonic regression per-kelas (di-fit di
validation set, bukan test set).

In [ ]:
# Reliability check BEFORE calibration: accuracy inside the "confident" zone (>0.85 or <0.15)
# that v3-as-run used to skip Tier-2 entirely.
conf_raw = p_t1_test.max(1)
confident_zone = (conf_raw > 0.85) | (conf_raw < 0.15)
acc_in_zone = accuracy_score(y_test[confident_zone], pred_t1[confident_zone]) if confident_zone.any() else float("nan")
print(f"Tier-1 raw confidence >0.85 or <0.15: {confident_zone.mean()*100:.1f}% of test set")
print(f"  Accuracy of Tier-1 ALONE inside that 'confident' zone: {acc_in_zone:.4f}")
print(f"  (this is exactly the subset v3-as-run's cascade let Tier-1 decide alone, skipping Tier-2)")

In [ ]:
def fit_isotonic_calibrators(y_ref, p_ref, n_classes):
    calibrators = []
    y_bin = label_binarize(y_ref, classes=list(range(n_classes)))
    for c in range(n_classes):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(p_ref[:, c], y_bin[:, c])
        calibrators.append(ir)
    return calibrators

def apply_isotonic(calibrators, p):
    out = np.column_stack([cal.predict(p[:, c]) for c, cal in enumerate(calibrators)])
    out = out / out.sum(axis=1, keepdims=True).clip(min=1e-9)
    return out

N_CLS = len(le.classes_)
cal_t1 = fit_isotonic_calibrators(y_val, p_t1_val, N_CLS)
p_t1_val_cal  = apply_isotonic(cal_t1, p_t1_val)
p_t1_test_cal = apply_isotonic(cal_t1, p_t1_test)

res_t1_cal = evaluate("Tier-1 XGBoost (tuned + lexical, calibrated)", y_test,
                      p_t1_test_cal.argmax(1), p_t1_test_cal)
RESULTS.append(res_t1_cal)

conf_cal = p_t1_test_cal.max(1)
zone_cal = (conf_cal > 0.85) | (conf_cal < 0.15)
acc_zone_cal = accuracy_score(y_test[zone_cal], p_t1_test_cal[zone_cal].argmax(1)) if zone_cal.any() else float("nan")
print(f"\nAfter calibration — confident zone size: {zone_cal.mean()*100:.1f}%  "
      f"accuracy inside zone: {acc_zone_cal:.4f}")
print("(compare this against the pre-calibration number above — this is the concrete evidence for")
print(" whether calibration alone fixes the cascade's over-trust problem, or whether the threshold")
print(" itself also needs to be re-tuned — both are done in §8.)")

---
## §6 — Tier 2: CNN + Self-Attention (Character-Level, PyTorch)

**Perubahan arsitektur dari v3:** ablation study v3-as-run sendiri menunjukkan **CNN-only ≥
CNN-BiLSTM** dalam akurasi (0.8268 vs 0.8203) sambil ~4× lebih cepat per epoch (~10s vs ~44s),
dan **LSTM-only** adalah model DL paling lambat (~72s/epoch) sekaligus paling lemah. Kesimpulannya:
komponen rekuren (BiLSTM) di v3 adalah *liability* — lebih lambat tanpa menambah akurasi.

v4 mengganti BiLSTM dengan **self-attention** (Transformer encoder ringan): tetap memberi
*global receptive field* melewati jendela lokal CNN, tapi sepenuhnya paralel (tidak ada
bottleneck sekuensial rekuren) — idealnya mendapat manfaat BiLSTM (konteks global) tanpa
ongkos kecepatannya. CNN-BiLSTM asli, CNN-only, dan LSTM-only tetap disimpan sebagai
pembanding ablasi di §10.

In [ ]:
URL_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    r".-_~:/?#[]@!$&'()*+,;=%"
)
CHAR2IDX  = {c: i + 2 for i, c in enumerate(URL_CHARS)}  # 0=PAD, 1=UNK
VOCAB_SIZE = len(CHAR2IDX) + 2
MAX_LEN    = 200  # see §0 validation cell for coverage vs this cutoff

def url_to_tensor(url):
    ids = [CHAR2IDX.get(c, 1) for c in str(url)[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

class URLDataset(Dataset):
    def __init__(self, urls, labels):
        self.urls = urls.reset_index(drop=True) if hasattr(urls, "reset_index") else pd.Series(list(urls)).reset_index(drop=True)
        self.labels = labels
    def __len__(self): return len(self.urls)
    def __getitem__(self, i): return url_to_tensor(self.urls[i]), int(self.labels[i])

BS = 256
train_ds = URLDataset(train_df["url"], y_train)
val_ds   = URLDataset(val_df["url"],   y_val)
test_ds  = URLDataset(test_df["url"],  y_test)
ood_ds   = URLDataset(ood_df["url"],   np.zeros(len(ood_df), int))

def mk_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BS, shuffle=shuffle, num_workers=2, pin_memory=DEVICE == "cuda")

train_loader = mk_loader(train_ds, True)
val_loader   = mk_loader(val_ds,   False)
test_loader  = mk_loader(test_ds,  False)
ood_loader   = mk_loader(ood_ds,   False)
print(f"Vocab size: {VOCAB_SIZE}  Max URL length: {MAX_LEN}")

In [ ]:
class CNNAttn(nn.Module):
    """Character-level CNN + Transformer self-attention (no recurrence).
    Replaces v3's CNN-BiLSTM: the recurrent block was slower AND not more
    accurate per v3's own ablation. This keeps CNN's local pattern detection
    (e.g. l->1 substitution, character n-grams) and adds a global receptive
    field via lightweight self-attention instead of a sequential BiLSTM."""
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filters=128, kernels=(3, 5, 7),
                 d_model=256, nhead=4, n_layers=2, n_cls=4, drop=0.3, max_len=MAX_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.pos = nn.Parameter(torch.zeros(1, max_len, emb))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(emb, filters, k, padding=k // 2),
                          nn.GELU(), nn.Dropout(drop / 2))
            for k in kernels])
        cnn_dim = filters * len(kernels)
        self.proj = nn.Linear(cnn_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=drop, activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(drop),
            nn.Linear(128, n_cls))

    def forward(self, x):
        pad_mask = (x == 0)                                        # B,L  True=PAD
        e  = self.embed(x) + self.pos[:, :x.size(1), :]            # B,L,E
        ci = e.permute(0, 2, 1)                                    # B,E,L
        c  = torch.cat([cv(ci).permute(0, 2, 1) for cv in self.convs], -1)  # B,L,Fc
        h  = self.proj(c)                                          # B,L,D
        h  = self.encoder(h, src_key_padding_mask=pad_mask)        # B,L,D
        w  = self.attn_pool(h).masked_fill(pad_mask.unsqueeze(-1), -1e4)
        w  = torch.softmax(w, 1)                                    # B,L,1
        p  = (h * w).sum(1)                                         # B,D
        return self.head(p)

model_dl = CNNAttn().to(DEVICE)
total = sum(p.numel() for p in model_dl.parameters())
print(f"CNN-Attn parameters: {total:,}")

In [ ]:
def train_dl(model, train_loader, val_loader, epochs=15, lr=3e-4, patience=6):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_vl, best_state, no_improve = float("inf"), None, 0
    tr_hist, vl_hist = [], []

    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(xb)
        model.eval(); vl = 0.0; correct = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits  = model(xb)
                vl     += crit(logits, yb).item() * len(xb)
                correct += (logits.argmax(1) == yb).sum().item()
        tl /= len(train_loader.dataset); vl /= len(val_loader.dataset)
        va  = correct / len(val_loader.dataset)
        tr_hist.append(tl); vl_hist.append(vl)
        sched.step()
        if vl < best_vl - 1e-5:
            best_vl = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        print(f"  Epoch {ep:02d}/{epochs} | TrLoss {tl:.4f} | ValLoss {vl:.4f} | ValAcc {va:.4f}"
              + (" [best]" if no_improve == 0 else ""))
        if no_improve >= patience:
            print(f"  Early stopping (no val improvement for {patience} epochs)")
            break
    model.load_state_dict(best_state)
    return tr_hist, vl_hist

print("Training CNN-Attn...")
tr_hist, vl_hist = train_dl(model_dl, train_loader, val_loader, epochs=18)

In [ ]:
# Training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tr_hist, label="Train Loss", color="#3498db")
ax.plot(vl_hist, label="Val Loss",   color="#e74c3c")
ax.set(title="Tier-2 CNN-Attn training curve", xlabel="epoch", ylabel="loss")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "tier2_training_curve.png", dpi=150)
plt.show()

In [ ]:
def dl_infer(model, loader):
    model.eval(); probas = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            p = torch.softmax(model(xb), dim=1).cpu().numpy()
            probas.append(p)
    probas = np.concatenate(probas, 0)
    return probas, probas.argmax(1)

p_t2_val, _      = dl_infer(model_dl, val_loader)
p_t2_test, pred_t2 = dl_infer(model_dl, test_loader)
res_t2 = evaluate("Tier-2 CNN-Attn (standalone, raw)", y_test, pred_t2, p_t2_test)
RESULTS.append(res_t2)

torch.save(model_dl.state_dict(), OUT_DIR / "tier2_cnnattn.pt")
with open(OUT_DIR / "tier2_arch.json", "w") as f:
    json.dump({"vocab": VOCAB_SIZE, "emb": 64, "filters": 128, "kernels": [3, 5, 7],
               "d_model": 256, "nhead": 4, "n_layers": 2, "n_cls": N_CLS,
               "drop": 0.3, "max_len": MAX_LEN}, f, indent=2)
print("Tier-2 CNN-Attn saved.")

In [ ]:
# Calibrate Tier-2 the same way as Tier-1
cal_t2 = fit_isotonic_calibrators(y_val, p_t2_val, N_CLS)
p_t2_val_cal  = apply_isotonic(cal_t2, p_t2_val)
p_t2_test_cal = apply_isotonic(cal_t2, p_t2_test)
res_t2_cal = evaluate("Tier-2 CNN-Attn (standalone, calibrated)", y_test,
                      p_t2_test_cal.argmax(1), p_t2_test_cal)
RESULTS.append(res_t2_cal)

---
## §7 — Stacker (Logistic Regression Meta-Learner)

Input diperluas dari v3 (`[xgb(4), dl(4)]` -> 8-dim) menjadi
`[xgb_cal(4), tfidf_linear(4), cnn_attn_cal(4)]` -> 12-dim, memberi stacker sinyal dari
3 keluarga model yang cukup independen (tabular gradient boosting, linear n-gram, deep
character sequence). Dilatih di **validation set**, dievaluasi di **test set**.

In [ ]:
STACKER_NAME = "Stacker (Tier1+Tier1b+Tier2, calibrated)"

X_stack_val  = np.hstack([p_t1_val_cal,  p_t1b_val,  p_t2_val_cal])   # (N_val, 12)
X_stack_test = np.hstack([p_t1_test_cal, p_t1b_test, p_t2_test_cal])  # (N_test, 12)

stacker = LogisticRegression(C=1.0, max_iter=500, random_state=SEED, n_jobs=-1)
stacker.fit(X_stack_val, y_val)

p_stack_test = stacker.predict_proba(X_stack_test)
pred_stack   = p_stack_test.argmax(1)
res_stack    = evaluate(STACKER_NAME, y_test, pred_stack, p_stack_test)
RESULTS.append(res_stack)

with open(OUT_DIR / "stacker.pkl", "wb") as f: pickle.dump(stacker, f)
print("Stacker saved.")

---
## §8 — Cascade System v4: Calibrated Gating + Tuned Thresholds

**Kenapa cascade v3-as-run gagal (evaluasi, dengan angka):** cascade men-*gate* pakai
`max(softmax)` Tier-1 **mentah** pada ambang tetap 0.85/0.15 yang diambil langsung dari dokumen
rencana, tidak pernah divalidasi di data ini. Karena Tier-1 sendiri hanya ~62% akurat dan
overconfident, ~46% traffic "diputuskan sendiri" oleh Tier-1 di zona itu — dan sebagian
signifikan dari keputusan "percaya diri" itu salah, sehingga cascade (Acc 0.7664) kalah jauh
dari stacker yang selalu menjalankan kedua tier (Acc 0.8329).

v4 memperbaiki ini dengan 2 perubahan:
1. Gate memakai **P(malicious) terkalibrasi** = `1 - P(benign)`, bukan `max(softmax)` mentah —
   ini juga selaras dengan tabel threshold aksi (block/review/allow) yang memang biner
   (malicious vs tidak), bukan argmax dari 4 kelas.
2. Ambang gate **di-tune di validation set** lewat pencarian grid, bukan konstanta tebakan.

In [ ]:
# Reproduce the v3-as-run cascade logic exactly (raw, uncalibrated, fixed 0.85/0.15 on
# argmax confidence) purely for an apples-to-apples historical comparison.
def cascade_predict_v3naive(feats_df, xgb_p_raw, dl_m, loader_fn, urls_series, stacker_m,
                            conf_high=0.85, conf_low=0.15):
    n = len(urls_series)
    xgb_c = xgb_p_raw.max(1)
    ambig = (xgb_c > conf_low) & (xgb_c < conf_high)
    dl_p_full = np.zeros((n, N_CLS))
    if ambig.any():
        ambig_urls = urls_series.reset_index(drop=True)[ambig].reset_index(drop=True)
        ambig_ds   = URLDataset(ambig_urls, np.zeros(ambig.sum(), int))
        ambig_ld   = mk_loader(ambig_ds, False)
        dl_p_ambig, _ = dl_infer(dl_m, ambig_ld)
        dl_p_full[ambig] = dl_p_ambig
    final_p = xgb_p_raw.copy()
    if ambig.any():
        # v3-as-run's stacker only took [xgb(4), dl(4)] -> reproduce that 8-dim shape here
        meta = np.hstack([xgb_p_raw[ambig], dl_p_full[ambig]])
        final_p[ambig] = stacker_v3_shape.predict_proba(meta)
    return final_p.argmax(1), final_p, ambig

# A stacker trained on the ORIGINAL 8-dim [xgb_raw, dl_raw] shape, just to replay v3's exact logic
stacker_v3_shape = LogisticRegression(C=1.0, max_iter=500, random_state=SEED, n_jobs=-1)
stacker_v3_shape.fit(np.hstack([p_t1_val, p_t2_val]), y_val)

pred_casc_v3naive, p_casc_v3naive, ambig_v3naive = cascade_predict_v3naive(
    X_test_lex, p_t1_test, model_dl, mk_loader, test_df["url"], stacker_v3_shape)
res_casc_v3naive = evaluate("Cascade (v3-style, raw conf, fixed 0.85/0.15)", y_test,
                            pred_casc_v3naive, p_casc_v3naive)
RESULTS.append(res_casc_v3naive)
print(f"\n[v3-style cascade] Tier-2 invoked on {ambig_v3naive.mean()*100:.1f}% of traffic")

In [ ]:
benign_idx = list(le.classes_).index("benign")

# Validate the binary "malicious" framing directly against ground-truth is_malicious
if "is_malicious" in val_df.columns:
    y_val_mal = val_df["is_malicious"].values.astype(int)
else:
    y_val_mal = (y_val != benign_idx).astype(int)

pmal_t1_val = 1 - p_t1_val_cal[:, benign_idx]
fpr_curve, tpr_curve, thr_curve = roc_curve(y_val_mal, pmal_t1_val)
prec_curve, rec_curve, thr_pr = precision_recall_curve(y_val_mal, pmal_t1_val)
print(f"Tier-1 calibrated P(malicious) vs ground-truth is_malicious — "
      f"ROC-AUC={roc_auc_score(y_val_mal, pmal_t1_val):.4f}")

In [ ]:
def sweep_cascade_thresholds(p_t1_cal, p_t1b, p_t2_cal, y_true, stacker_m,
                             low_grid, high_grid):
    """Simulate cascade routing on validation data for a grid of
    (low, high) gates on Tier-1's calibrated P(malicious), reporting the
    accuracy/F1 vs. % of traffic routed to Tier-2 trade-off."""
    pmal = 1 - p_t1_cal[:, benign_idx]
    rows = []
    for low in low_grid:
        for high in high_grid:
            if high <= low:
                continue
            ambig = (pmal > low) & (pmal < high)
            final_p = p_t1_cal.copy()
            if ambig.any():
                meta = np.hstack([p_t1_cal[ambig], p_t1b[ambig], p_t2_cal[ambig]])
                final_p[ambig] = stacker_m.predict_proba(meta)
            pred = final_p.argmax(1)
            rows.append({
                "low": round(low, 2), "high": round(high, 2),
                "tier2_pct": ambig.mean() * 100,
                "accuracy": accuracy_score(y_true, pred),
                "f1_weighted": f1_score(y_true, pred, average="weighted"),
            })
    return pd.DataFrame(rows)

sweep_df = sweep_cascade_thresholds(
    p_t1_val_cal, p_t1b_val, p_t2_val_cal, y_val, stacker,
    low_grid=[0.02, 0.05, 0.10, 0.15, 0.20, 0.30],
    high_grid=[0.50, 0.60, 0.70, 0.80, 0.90, 0.95])

full_stacker_val_acc = accuracy_score(
    y_val, stacker.predict(np.hstack([p_t1_val_cal, p_t1b_val, p_t2_val_cal])))
print(f"Full stacker accuracy on VAL (always run both tiers): {full_stacker_val_acc:.4f}\n")

# Recommended operating point: fewest Tier-2 calls while staying within 1pp of the
# full-stacker accuracy on validation data.
candidates = sweep_df[sweep_df["accuracy"] >= full_stacker_val_acc - 0.01].sort_values("tier2_pct")
print("Top 10 candidate thresholds (accuracy within 1pp of full stacker, sorted by Tier-2 usage):")
print(candidates.head(10).to_string(index=False))

In [ ]:
if len(candidates) > 0:
    best_row = candidates.iloc[0]
    CASC_LOW, CASC_HIGH = float(best_row["low"]), float(best_row["high"])
    print(f"Selected cascade v4 gate: P(malicious) in ({CASC_LOW}, {CASC_HIGH}) => Tier-2 invoked")
    print(f"  Expected Tier-2 usage: {best_row['tier2_pct']:.1f}%  (val accuracy {best_row['accuracy']:.4f} "
          f"vs full-stacker {full_stacker_val_acc:.4f})")
else:
    # No threshold gets within 1pp of the full stacker without near-100% Tier-2 usage —
    # be honest about it rather than force a bad default.
    CASC_LOW, CASC_HIGH = 0.0, 1.0
    print("WARNING: no threshold band keeps cascade within 1pp of the full stacker's accuracy")
    print("without routing nearly all traffic to Tier-2. This means Tier-1 alone (lexical-only)")
    print("is not reliable enough to safely skip Tier-2 for a meaningful share of traffic on this")
    print("dataset. Recommendation: run Tier-2 (CNN-Attn) on effectively all traffic that passes")
    print("Tier-0, and reserve 'cascade savings' for literal blocklist/allowlist hits only (see the")
    print("Tier-0 simulation below) rather than a confidence-based skip of Tier-2.")
    print("Falling back to (0.0, 1.0), i.e. Tier-2 is invoked on 100% of traffic for the v4 cascade.")

In [ ]:
def cascade_predict_v4(urls_series, p_t1_cal, p_t1b, dl_m, stacker_m, low, high):
    n = len(urls_series)
    pmal = 1 - p_t1_cal[:, benign_idx]
    ambig = (pmal > low) & (pmal < high)

    dl_p_full = np.zeros((n, N_CLS))
    if ambig.any():
        ambig_urls = urls_series.reset_index(drop=True)[ambig].reset_index(drop=True)
        ambig_ds   = URLDataset(ambig_urls, np.zeros(ambig.sum(), int))
        ambig_ld   = mk_loader(ambig_ds, False)
        dl_p_raw, _ = dl_infer(dl_m, ambig_ld)
        dl_p_full[ambig] = apply_isotonic(cal_t2, dl_p_raw)

    final_p = p_t1_cal.copy()
    if ambig.any():
        meta = np.hstack([p_t1_cal[ambig], p_t1b[ambig], dl_p_full[ambig]])
        final_p[ambig] = stacker_m.predict_proba(meta)

    return final_p.argmax(1), final_p, ambig

pred_casc_v4, p_casc_v4, ambig_v4 = cascade_predict_v4(
    test_df["url"], p_t1_test_cal, p_t1b_test, model_dl, stacker, CASC_LOW, CASC_HIGH)
res_casc_v4 = evaluate("Cascade v4 (calibrated P(malicious), tuned gate)", y_test, pred_casc_v4, p_casc_v4)
RESULTS.append(res_casc_v4)
print(f"\n[Cascade v4] Tier-2 invoked on {ambig_v4.mean()*100:.1f}% of test traffic "
      f"(vs {ambig_v3naive.mean()*100:.1f}% for the v3-style cascade)")

In [ ]:
print("="*70)
print("CASCADE COMPARISON — TEST SET")
print("="*70)
cmp_cols = ["Accuracy", "F1-weighted", "PR-AUC", "FPR"]
cmp_df = pd.DataFrame(RESULTS).set_index("Model").loc[
    [STACKER_NAME, "Cascade (v3-style, raw conf, fixed 0.85/0.15)",
     "Cascade v4 (calibrated P(malicious), tuned gate)"], cmp_cols].round(4)
print(cmp_df.to_string())

---
## §9 — Domain-Leakage Check & Tier-0 Blocklist/Allowlist Simulation

Dataset ini mengklaim split *"domain-grouped, zero-leakage"*. §0 sudah menghitung overlap
domain train/test secara empiris. Di sini kita pakai angka itu untuk dua hal:

1. **Uji generalisasi lebih ketat**: metrik model final dihitung ulang khusus untuk baris test
   yang domainnya **tidak pernah muncul** di train — proxy yang lebih jujur untuk performa di
   domain benar-benar baru (yang mana justru skenario paling realistis untuk phishing baru).
2. **Simulasi Tier-0**: di produksi nyata, domain yang berulang itu justru *aset* — begitu satu
   URL dari domain tertentu diverifikasi, seluruh domain itu bisa langsung masuk cache
   blocklist/allowlist tanpa perlu model sama sekali untuk kunjungan berikutnya.

In [ ]:
print(f"Test rows sharing a registered_domain with train: {TEST_OVERLAP_MASK.mean()*100:.2f}%")

novel_mask = ~TEST_OVERLAP_MASK
if novel_mask.sum() > 20:
    print(f"\nStricter generalization check — accuracy on the {novel_mask.sum():,} test rows "
          f"whose domain is genuinely NEW (not seen in train):")
    for name, pred in [(STACKER_NAME, pred_stack), ("Cascade v4", pred_casc_v4)]:
        acc_novel = accuracy_score(y_test[novel_mask], pred[novel_mask])
        acc_all   = accuracy_score(y_test, pred)
        print(f"  {name:45s}  all={acc_all:.4f}   novel-domain-only={acc_novel:.4f}")
else:
    print("Too few (or zero) overlapping-domain rows to compute a meaningful split — "
          "consistent with the dataset's 'zero-leakage' claim.")

In [ ]:
# Tier-0 simulation: build a domain reputation cache from TRAIN, where a domain is only
# cached if its labels in train are highly consistent (>=95% one label) — mimicking a
# blocklist/allowlist populated from verified historical decisions.
if DOMAIN_COL:
    dom_label_purity = train_df.groupby(DOMAIN_COL)["label"].agg(
        lambda s: s.value_counts(normalize=True).iloc[0])
    dom_majority_label = train_df.groupby(DOMAIN_COL)["label"].agg(lambda s: s.value_counts().index[0])
    CACHE = dom_majority_label[dom_label_purity >= 0.95].to_dict()
    print(f"Tier-0 cache size (domains with >=95% label purity in train): {len(CACHE):,}")

    test_domains_lower = test_df[DOMAIN_COL].str.lower()
    cache_hit = test_domains_lower.isin(CACHE.keys()).values
    print(f"Tier-0 cache HIT rate on test traffic: {cache_hit.mean()*100:.1f}%  "
          f"(these would get an instant verdict, no ML tiers needed)")

    if cache_hit.any():
        cache_pred_labels = test_domains_lower[cache_hit].map(CACHE)
        cache_pred_enc = le.transform(cache_pred_labels)
        cache_acc = accuracy_score(y_test[cache_hit], cache_pred_enc)
        print(f"  Accuracy of Tier-0 cache verdicts alone: {cache_acc:.4f}")
        print(f"  => Tier-0 alone can safely resolve {cache_hit.mean()*100:.1f}% of traffic instantly;")
        print(f"     the remaining {100-cache_hit.mean()*100:.1f}% still needs Tier-1/2/stacker.")
else:
    print("registered_domain column not available — Tier-0 simulation skipped.")

---
## §10 — Latency Benchmark (CPU, single-URL — realistic online-serving estimate)

In [ ]:
def bench_tier1(n=500):
    idx = np.random.RandomState(SEED).choice(len(X_te), size=min(n, len(X_te)), replace=False)
    times = []
    for i in idx:
        row = X_te[i:i+1]
        t0 = time.perf_counter()
        tier1_xgb.predict(xgb.DMatrix(row))
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

def bench_tier1b(n=500):
    urls = test_df["url"].astype(str).values
    idx = np.random.RandomState(SEED).choice(len(urls), size=min(n, len(urls)), replace=False)
    times = []
    for i in idx:
        t0 = time.perf_counter()
        tier1b.predict_proba(tfidf.transform([urls[i]]))
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

def bench_tier2_cpu(n=300):
    model_cpu = CNNAttn().to("cpu")
    model_cpu.load_state_dict(model_dl.state_dict())
    model_cpu.eval()
    times = []
    with torch.no_grad():
        for i in range(min(n, len(test_ds))):
            xb, _ = test_ds[i]
            xb = xb.unsqueeze(0)
            t0 = time.perf_counter()
            model_cpu(xb)
            times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

t1_times  = bench_tier1()
t1b_times = bench_tier1b()
t2_times  = bench_tier2_cpu()

def pct(a, p): return float(np.percentile(a, p))

print("Latency benchmark — single URL, CPU inference (worst-case online-serving estimate):")
print(f"  Tier 1  (XGBoost)        p50={pct(t1_times,50):6.2f}ms  p95={pct(t1_times,95):6.2f}ms")
print(f"  Tier 1b (TF-IDF+SGD)     p50={pct(t1b_times,50):6.2f}ms  p95={pct(t1b_times,95):6.2f}ms")
print(f"  Tier 2  (CNN-Attn, CPU)  p50={pct(t2_times,50):6.2f}ms  p95={pct(t2_times,95):6.2f}ms")

tier1_p95 = pct(t1_times, 95) + pct(t1b_times, 95)
full_p95  = tier1_p95 + pct(t2_times, 95)
print(f"\n  v3 target — Tier-1 path p95 < 20ms   -> {'PASS' if tier1_p95 < 20 else 'FAIL'} ({tier1_p95:.1f}ms)")
print(f"  v3 target — full path (0->1->2) p95 < 200ms -> {'PASS' if full_p95 < 200 else 'FAIL'} ({full_p95:.1f}ms)")
print("\n  Note: if Tier-2 alone is already well under budget even at 100% traffic, the whole")
print("  premise of skipping it via confidence-gating (§8) may not even be necessary from a")
print("  latency standpoint — only from a raw compute-cost standpoint at very high QPS.")

---
## §11 — Action Thresholds (Block / Review / Allow) — Re-derived, not Guessed

v3's threshold table (≥0.85 block, 0.6–0.85 review, <0.6 allow) was **specified in the planning
doc before any data existed**. Now that we have calibrated probabilities and ground truth, we
derive the block-threshold empirically to hit the **FPR < 2%** target from the metrics table,
using the stacker's calibrated `P(malicious)` on the **validation set**, then verify on test.

In [ ]:
def find_threshold_for_fpr(y_true_mal, p_mal, target_fpr):
    fpr, tpr, thr = roc_curve(y_true_mal, p_mal)
    valid = fpr <= target_fpr
    if valid.any():
        idx = np.where(valid)[0][-1]
    else:
        idx = 0
    return float(thr[idx]), float(fpr[idx]), float(tpr[idx])

pmal_stack_val = 1 - stacker.predict_proba(X_stack_val)[:, benign_idx]
thr_block, fpr_block, tpr_block = find_threshold_for_fpr(y_val_mal, pmal_stack_val, target_fpr=0.02)
thr_review, fpr_review, tpr_review = find_threshold_for_fpr(y_val_mal, pmal_stack_val, target_fpr=0.10)

print(f"Block-auto threshold  (target FPR<=2%):  P(malicious) >= {thr_block:.3f}  "
      f"(val FPR={fpr_block:.4f}, recall={tpr_block:.4f})")
print(f"Block-review threshold (target FPR<=10%): P(malicious) >= {thr_review:.3f}  "
      f"(val FPR={fpr_review:.4f}, recall={tpr_review:.4f})")

# Verify on TEST
pmal_stack_test = 1 - p_stack_test[:, benign_idx]
y_test_mal = test_df["is_malicious"].values.astype(int) if "is_malicious" in test_df.columns \
    else (y_test != benign_idx).astype(int)
pred_mal_test = (pmal_stack_test >= thr_block).astype(int)
test_fpr = ((pred_mal_test == 1) & (y_test_mal == 0)).sum() / max((y_test_mal == 0).sum(), 1)
test_recall = ((pred_mal_test == 1) & (y_test_mal == 1)).sum() / max((y_test_mal == 1).sum(), 1)
print(f"\nOn TEST with block_auto threshold: FPR={test_fpr:.4f}  Recall(malicious)={test_recall:.4f}")

ACTION_THRESHOLDS_V4 = {"block_auto": thr_block, "block_review": thr_review, "allow_log": 0.0}
print("\nRe-derived action thresholds (v4):", {k: round(v, 3) for k, v in ACTION_THRESHOLDS_V4.items()})
print("v3 planning-doc thresholds (never validated on data): {'block_auto': 0.85, 'block_review': 0.6}")

---
## §12 — Ablation Study: DL Architectures

Melatih ulang 3 arsitektur pembanding di data & fitur yang sama, untuk memverifikasi ulang
temuan v3-as-run (CNN-only ≥ CNN-BiLSTM, LSTM-only terlemah & terlambat) dan melihat di mana
`CNNAttn` (Tier-2 v4) berdiri relatif terhadap semuanya.

In [ ]:
class CNNOnly(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filt=256, n_cls=4, drop=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, filt, k, padding=k // 2) for k in [3, 5, 7]])
        self.head  = nn.Sequential(nn.Dropout(drop), nn.Linear(filt * 3, 128),
                                   nn.GELU(), nn.Dropout(drop), nn.Linear(128, n_cls))
    def forward(self, x):
        e = self.embed(x).permute(0, 2, 1)
        p = torch.cat([F.adaptive_max_pool1d(F.gelu(c(e)), 1).squeeze(-1) for c in self.convs], 1)
        return self.head(p)

class LSTMOnly(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, emb=64, h=256, n_cls=4, drop=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm  = nn.LSTM(emb, h, 2, bidirectional=True, batch_first=True, dropout=drop)
        self.head  = nn.Sequential(nn.Dropout(drop), nn.Linear(h * 2, n_cls))
    def forward(self, x):
        _, (h, _) = self.lstm(self.embed(x))
        return self.head(torch.cat([h[-1], h[-2]], 1))

class CNNBiLSTM_v3(nn.Module):
    """Original v3 architecture, kept purely as an ablation reference point."""
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filters=128, kernels=(3, 5, 7),
                 lstm_h=128, lstm_l=2, n_cls=4, drop=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(emb, filters, k, padding=k // 2), nn.GELU(), nn.Dropout(drop / 2))
            for k in kernels])
        cnn_dim = filters * len(kernels)
        self.bilstm = nn.LSTM(cnn_dim, lstm_h, lstm_l, bidirectional=True, batch_first=True,
                              dropout=drop if lstm_l > 1 else 0.0)
        self.attn = nn.Linear(lstm_h * 2, 1)
        self.head = nn.Sequential(nn.Linear(lstm_h * 2, 128), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(128, n_cls))
    def forward(self, x):
        e  = self.embed(x)
        ci = e.permute(0, 2, 1)
        c  = torch.cat([cv(ci).permute(0, 2, 1) for cv in self.convs], -1)
        out, _ = self.bilstm(c)
        w = torch.softmax(self.attn(out), 1)
        p = (out * w).sum(1)
        return self.head(p)

for name, m in [("CNN-only", CNNOnly()), ("LSTM-only", LSTMOnly()), ("CNN-BiLSTM (v3 arch)", CNNBiLSTM_v3())]:
    m = m.to(DEVICE)
    print(f"\nTraining {name}...")
    train_dl(m, train_loader, val_loader, epochs=12, patience=5)
    p, pred = dl_infer(m, test_loader)
    RESULTS.append(evaluate(f"Ablation: {name}", y_test, pred, p))

In [ ]:
# Ablation summary table
ablation_rows = ["Tier-2 CNN-Attn (standalone, calibrated)", "Ablation: CNN-only",
                 "Ablation: LSTM-only", "Ablation: CNN-BiLSTM (v3 arch)"]
res_df_tmp = pd.DataFrame(RESULTS).set_index("Model")
present = [r for r in ablation_rows if r in res_df_tmp.index]
fig, ax = plt.subplots(figsize=(9, 4))
res_df_tmp.loc[present, "Accuracy"].plot(kind="barh", ax=ax, color="#9b59b6")
ax.set(title="DL Architecture Ablation — Test Accuracy", xlabel="Accuracy")
plt.tight_layout()
plt.savefig(OUT_DIR / "ablation_comparison.png", dpi=150)
plt.show()
print(res_df_tmp.loc[present, ["Accuracy", "F1-weighted", "PR-AUC", "FPR"]].round(4).to_string())

---
## §13 — Segmented Evaluation + Statistical Tests

In [ ]:
# Full results table
res_df = pd.DataFrame(RESULTS).set_index("Model").round(4)
print("\n" + "=" * 70)
print("COMPREHENSIVE MODEL COMPARISON -- TEST SET")
print("=" * 70)
print(res_df.to_string())

In [ ]:
plot_cols = [c for c in ["Accuracy", "F1-weighted", "F1-macro", "PR-AUC", "FPR"] if c in res_df.columns]
fig, ax = plt.subplots(figsize=(12, len(res_df) * 0.5 + 2))
sns.heatmap(res_df[plot_cols].astype(float), annot=True, fmt=".4f", cmap="RdYlGn",
           cbar_kws={"label": "score"}, ax=ax)
ax.set(title="All Models — Test Set Comparison")
plt.tight_layout()
plt.savefig(OUT_DIR / "all_models_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Per-class PR-AUC / FPR breakdown — the FINAL production candidate (choose one; default: stacker)
FINAL_MODEL_NAME = STACKER_NAME
final_pred, final_p = pred_stack, p_stack_test

print(f"Per-class Evaluation -- {FINAL_MODEL_NAME}:")
y_bin = label_binarize(y_test, classes=list(range(N_CLS)))
for i, cls in enumerate(le.classes_):
    pr_auc = average_precision_score(y_bin[:, i], final_p[:, i])
    fp_i = ((final_pred == i) & (y_test != i)).sum()
    tn_i = ((final_pred != i) & (y_test != i)).sum()
    fpr_i = fp_i / (fp_i + tn_i + 1e-9)
    n_cls_i = (y_test == i).sum()
    print(f"  [{cls:8s}] n={n_cls_i:5,}  PR-AUC={pr_auc:.4f}  FPR={fpr_i:.4f}")

In [ ]:
# McNemar's test: Cascade v4 vs Stacker, and Cascade v4 vs v3-style cascade
def mcnemar_report(name_a, pred_a, name_b, pred_b):
    c_a = (pred_a == y_test); c_b = (pred_b == y_test)
    n11 = (c_a & c_b).sum(); n10 = (c_a & ~c_b).sum()
    n01 = (~c_a & c_b).sum(); n00 = (~c_a & ~c_b).sum()
    result = mcnemar_test([[n11, n10], [n01, n00]], exact=False, correction=True)
    print(f"McNemar's Test -- {name_a} vs {name_b}")
    print(f"  Both correct: {n11:,}   Only {name_a}: {n10:,}   Only {name_b}: {n01:,}   Both wrong: {n00:,}")
    print(f"  Chi2: {result.statistic:.4f}  p-value: {result.pvalue:.6f}  "
          f"-> {'SIGNIFICANT' if result.pvalue < 0.05 else 'not significant'} (p<0.05)\n")

mcnemar_report("Cascade v4", pred_casc_v4, STACKER_NAME.split(' (')[0], pred_stack)
mcnemar_report("Cascade v4", pred_casc_v4, "Cascade v3-style", pred_casc_v3naive)

In [ ]:
# Per-source breakdown
if "source" in test_df.columns:
    print("Per-source F1 (final model):")
    for src in sorted(test_df["source"].unique()):
        mask = test_df["source"].values == src
        if mask.sum() < 10:
            continue
        f1 = f1_score(y_test[mask], final_pred[mask], average="weighted", zero_division=0)
        print(f"  [{src:30s}]  n={mask.sum():6,}  F1={f1:.4f}")

# Per high-risk-TLD-bucket breakdown (new in v4)
tld_bucket = X_test_lex["tld_risk"].values.astype(bool)
for name, mask in [("high-risk TLD", tld_bucket), ("normal TLD", ~tld_bucket)]:
    if mask.sum() < 10:
        continue
    f1 = f1_score(y_test[mask], final_pred[mask], average="weighted", zero_division=0)
    print(f"  [{name:14s}]  n={mask.sum():6,}  F1={f1:.4f}")

---
## §14 — OOD Evaluation: Indonesian Threat Case Study (Zero-Shot)

**Ini adalah tes paling penting di seluruh notebook.** v3-as-run gagal total di sini: dari 404 URL
judol/typosquat Indonesia zero-shot, model **tidak pernah memprediksi kelas "other" sekalipun**
(0/404), padahal 350/404 ground truth-nya memang "other". Headline metric di bawah langsung
mengecek apakah leksikon judol (§2) memperbaiki ini secara nyata, bukan cuma metrik agregat.

In [ ]:
print("OOD sample preview:")
print(ood_df.head(10))
print("\nOOD label distribution:")
if "label" in ood_df.columns:
    print(ood_df["label"].value_counts())

In [ ]:
d_ood = xgb.DMatrix(X_ood_lex[LEX_COLS].fillna(-1).values)
p_t1_ood_raw = tier1_xgb.predict(d_ood).reshape(-1, N_CLS)
p_t1_ood_cal = apply_isotonic(cal_t1, p_t1_ood_raw)

p_t2_ood_raw, _ = dl_infer(model_dl, ood_loader)
p_t2_ood_cal = apply_isotonic(cal_t2, p_t2_ood_raw)

p_stack_ood = stacker.predict_proba(np.hstack([p_t1_ood_cal, p_t1b_ood, p_t2_ood_cal]))
pred_ood    = p_stack_ood.argmax(1)
conf_ood    = p_stack_ood.max(1)
label_ood   = le.inverse_transform(pred_ood)

print("OOD Predicted class distribution (v4):")
print(pd.Series(label_ood).value_counts())
print(f"\nMean confidence : {conf_ood.mean():.4f}")
print(f"High conf (>=block_auto={ACTION_THRESHOLDS_V4['block_auto']:.2f}): "
      f"{(1-p_stack_ood[:,benign_idx] >= ACTION_THRESHOLDS_V4['block_auto']).mean()*100:.1f}%  (would auto-block)")

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

print("=" * 70)
print("HEADLINE FIX CHECK -- 'other' (judol) class recall on OOD")
print("=" * 70)
if "label" in ood_df.columns:
    other_idx = list(le.classes_).index("other")
    true_other = (ood_df["label"].values == "other")
    n_other = int(true_other.sum())
    if n_other > 0:
        k_other = int((pred_ood[true_other] == other_idx).sum())
        recall_other = k_other / n_other
        lo, hi = wilson_ci(k_other, n_other)
        print(f"  v4 recall on true-'other' OOD rows: {recall_other:.3f}  "
              f"(95% CI [{lo:.3f}, {hi:.3f}], n={n_other}, {k_other}/{n_other} correctly flagged)")
        print(f"  v3-as-run baseline for this exact metric: 0.000  (0/{n_other} predicted as 'other')")
    for cls in le.classes_:
        mask = ood_df["label"].values == cls
        n_c = int(mask.sum())
        if n_c == 0:
            continue
        cls_idx = list(le.classes_).index(cls)
        k_c = int((pred_ood[mask] == cls_idx).sum())
        lo_c, hi_c = wilson_ci(k_c, n_c)
        print(f"  [{cls:8s}] n={n_c:4d}  recall={k_c/n_c:.3f}  95% CI [{lo_c:.3f},{hi_c:.3f}]"
              + ("   (small-n: read with caution)" if n_c < 30 else ""))
else:
    print("No 'label' column in OOD set — skipping per-class recall.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(conf_ood, bins=25, color="#e74c3c", alpha=0.85, edgecolor="white")
ax.axvline(ACTION_THRESHOLDS_V4["block_auto"], color="white", lw=2, ls="--", label="Block-auto (v4)")
ax.axvline(ACTION_THRESHOLDS_V4["block_review"], color="yellow", lw=2, ls="--", label="Block-review (v4)")
ax.set(title="OOD Confidence Distribution\n(Indonesian Threats -- Zero-Shot)",
      xlabel="Model Confidence (max class proba)", ylabel="Count")
ax.legend()

ax = axes[1]
counts = pd.Series(label_ood).value_counts().reindex(LABEL_ORDER, fill_value=0)
bars = ax.bar(counts.index, counts.values, color=["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"])
for bar, v in zip(bars, counts.values):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
               f"{v}\n({v/len(label_ood)*100:.1f}%)", ha="center", fontsize=10)
ax.set(title="OOD Predicted Classes (v4, Zero-Shot)", ylabel="Count")

plt.suptitle("OOD Generalization: Indonesian Judol & Bank Typosquat URLs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "ood_evaluation.png", dpi=150)
plt.show()

In [ ]:
if "is_malicious" in ood_df.columns:
    y_ood_bin  = ood_df["is_malicious"].values.astype(int)
    detected   = (pred_ood != benign_idx).astype(int)
    detection_r = (detected == y_ood_bin).mean()
    recall_thr  = ((detected == 1) & (y_ood_bin == 1)).sum() / max(y_ood_bin.sum(), 1)
    lo_d, hi_d = wilson_ci(int(((detected == 1) & (y_ood_bin == 1)).sum()), int(y_ood_bin.sum()))
    print(f"OOD Detection Rate (binary malicious vs benign) : {detection_r:.4f}")
    print(f"OOD Threat Recall (zero-shot)                    : {recall_thr:.4f}  "
          f"95% CI [{lo_d:.3f},{hi_d:.3f}]")
    print(f"\nv3-as-run baseline: Detection Rate 0.3713, Threat Recall 0.3022")
else:
    top = pd.DataFrame({"url": ood_df["url"].values, "predicted": label_ood, "confidence": conf_ood}
                       ).sort_values("confidence", ascending=False)
    print("Top 15 OOD predictions by confidence:")
    print(top.head(15).to_string(index=False))

---
## §15 — Production Readiness Scorecard & Model Export

Skor akhir dibandingkan langsung terhadap tabel target dari dokumen rencana v3. Kolom yang FAIL
**bukan berarti pipeline-nya salah** — untuk konteks kenapa, baca catatan jujur setelah tabel.

In [ ]:
TARGETS = {
    "F1-weighted": (">=", 0.95),
    "PR-AUC":      (">=", 0.95),
    "FPR":         ("<",  0.02),
    "Precision":   (">=", 0.96),
    "Recall":      (">=", 0.93),
}

print(f"Production-readiness scorecard vs v3 targets -- model: {FINAL_MODEL_NAME}\n")
row = res_df.loc[FINAL_MODEL_NAME]
n_pass = 0
for metric, (op, target) in TARGETS.items():
    if metric not in row:
        continue
    val = row[metric]
    passed = (val >= target) if op == ">=" else (val < target)
    n_pass += int(passed)
    print(f"  {metric:14s} target {op} {target:<6}  actual={val:.4f}   {'PASS' if passed else 'FAIL'}")
print(f"\n  {n_pass}/{len(TARGETS)} aggregate targets met.")
print(f"\n  Latency targets: see §10 output above.")
print(f"  OOD 'other'(judol) recall: see §14 headline check above (v3 baseline was 0.000).")

### Cara membaca scorecard di atas (baca ini, jangan cuma lihat PASS/FAIL)

- Target v3 (F1≥95%, FPR<2%, dst) ditulis **sebelum ada data**, di dokumen perencanaan. Setelah
  benar-benar melatih di data URL-AF ini, target itu kemungkinan besar **tidak realistis** untuk
  dicapai murni dari sinyal lexical + karakter URL — bukan karena pipeline-nya kurang bagus, tapi
  karena membedakan phishing/malware/judol/benign hanya dari string URL memang punya batas
  informasi. Menaikkan compute atau menambah epoch tidak akan menembus batas ini.
- Yang benar-benar bisa mendorong ke arah target v3 adalah **sinyal yang tidak ada di dataset
  statis ini**: reputasi domain real-time (age, SSL, ASN), dan **lebih banyak & lebih beragam
  data judol** untuk generalisasi OOD (404 sampel zero-shot dengan hanya 14 phishing/40 benign
  itu jauh dari cukup untuk klaim recall yang presisi — lihat interval kepercayaan Wilson di §14).
- **Rekomendasi konkret untuk fase berikutnya** (di luar scope notebook training ini):
  1. Bangun pipeline fitur reputasi real-time di **serving side** (bukan training) — domain age,
     SSL issuer trust, redirect chain — lalu gabungkan lewat `apply_realtime_overrides()` yang
     diekspor di bawah, sesuai pola hard-override v3.
  2. Kumpulkan lebih banyak URL judol berlabel dari sumber beragam (bukan hanya pola
     `slot88.xyz`-style) supaya OOD recall untuk kelas "other" lebih meyakinkan.
  3. Set up model-drift monitoring (KL-divergence prediksi) di produksi — ini tidak bisa diuji
     dari notebook training statis, harus jalan di traffic live.
  4. Pertimbangkan **tidak** melakukan cascade skip berbasis confidence sama sekali jika angka
     latency Tier-2 di §10 sudah di bawah budget — jalankan semua tier untuk semua traffic
     (paling akurat), dan sisakan "cascade savings" murni untuk Tier-0 blocklist/allowlist
     (§9), yang **memang** aman untuk short-circuit karena berbasis histori terverifikasi,
     bukan tebakan confidence model.

In [ ]:
# Label encoder
with open(OUT_DIR / "label_encoder.json", "w") as f:
    json.dump({"classes": le.classes_.tolist(),
              "mapping": dict(zip(le.classes_, le.transform(le.classes_).tolist()))}, f)

# Feature columns + lexicons (single source of truth bundle for the serving team)
with open(OUT_DIR / "feature_cols.json", "w") as f:
    json.dump({"feature_cols": LEX_COLS, "brands": BRANDS,
              "high_risk_tlds": sorted(HIGH_RISK_TLDS), "suspicious_kw": SUSPICIOUS_KW,
              "judol_kw": JUDOL_KW, "shorteners": sorted(SHORTENERS)}, f, indent=2)

# Calibration objects
with open(OUT_DIR / "calibrators.pkl", "wb") as f:
    pickle.dump({"tier1": cal_t1, "tier2": cal_t2}, f)

# Cascade / action-threshold config (v4, empirically derived — see the sweep in Section 8)
cfg = {
    "cascade_gate_pmalicious_low": CASC_LOW,
    "cascade_gate_pmalicious_high": CASC_HIGH,
    "cascade_note": "Gate is on CALIBRATED P(malicious) = 1 - P(benign), tuned on val (Section 8), "
                   "NOT a fixed constant like v3's 0.85/0.15 on raw argmax confidence.",
    "action_thresholds": ACTION_THRESHOLDS_V4,
    "action_thresholds_v3_reference_only": {"block_auto": 0.85, "block_review": 0.60, "allow_log": 0.0},
    "vocab_size": VOCAB_SIZE, "max_url_len": MAX_LEN,
    "char2idx": CHAR2IDX, "label_order": LABEL_ORDER,
    "tier0_cache_size": len(CACHE) if DOMAIN_COL else 0,
}
with open(OUT_DIR / "cascade_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

# Full results
res_df.to_csv(OUT_DIR / "results_full.csv")
print("Core artifacts saved.")

In [ ]:
# Consolidated feature_extractor.py -- single source of truth for training AND serving.
# Built with inspect.getsource() from the exact functions used above, so there is no risk
# of a hand-copied serving version silently drifting from what was actually trained on
# (the training/serving skew problem the original v3 plan explicitly wanted to avoid).
import inspect

header_lines = [
    "# AFFERENT v4 -- feature_extractor.py",
    "# Auto-generated from the training notebook to guarantee training/serving parity.",
    "# Do not hand-edit the constants below -- regenerate from the notebook instead.",
    "import re, math",
    "from collections import Counter",
    "from urllib.parse import urlparse",
    "import tldextract",
    "",
    f"BRANDS = {BRANDS!r}",
    f"HIGH_RISK_TLDS = {HIGH_RISK_TLDS!r}",
    f"SUSPICIOUS_KW = {SUSPICIOUS_KW!r}",
    f"JUDOL_KW = {JUDOL_KW!r}",
    f"SHORTENERS = {SHORTENERS!r}",
    "",
]

func_blocks = [inspect.getsource(fn) for fn in
              [levenshtein, brand_min_distance, brand_domain_mismatch,
               shannon_entropy, max_consecutive_consonants, extract_features]]

override_lines = [
    "",
    "def apply_realtime_overrides(base_prediction, live_features):",
    "    # STUB for the serving team -- NOT implemented or trained here.",
    "    # live_features is computed live by the backend (domain_age_days, ssl_valid,",
    "    # ssl_trust_score, redirect_count, asn_reputation, ...) -- these are point-in-time",
    "    # facts that cannot be reconstructed retroactively from a static training set.",
    "    # Example hard-override ported from the v3 plan (only meaningful with a REAL",
    "    # domain_age -- the v3-as-run stub of -1 for every row meant this never fired):",
    "    #   domain_age = live_features.get('domain_age_days', 999)",
    "    #   lev_min = base_prediction.get('levenshtein_min', 99)",
    "    #   if domain_age < 3 and lev_min <= 2:",
    "    #       base_prediction['label'] = 'phishing'",
    "    #       base_prediction['confidence'] = max(base_prediction['confidence'], 0.70)",
    "    return base_prediction",
    "",
]

full_src = "\n".join(header_lines) + "\n\n" + "\n\n".join(func_blocks) + "\n" + "\n".join(override_lines)
with open(OUT_DIR / "feature_extractor.py", "w") as f:
    f.write(full_src)
print("Exported consolidated feature_extractor.py for serving-side reuse.")

print("\nAll artifacts saved to:", OUT_DIR)
for fp in sorted(OUT_DIR.iterdir()):
    print(f"  {fp.name:32s} {fp.stat().st_size/1024:8.1f} KB")

In [ ]:
print("\n" + "=" * 70)
print("AFFERENT v4 -- FINAL TRAINING SUMMARY")
print("=" * 70)
key = ["Logistic Regression", "Random Forest (standalone)",
      "XGBoost Standalone (lexical-only)", "Tier-1 XGBoost (tuned + lexical, calibrated)",
      "Tier-2 CNN-Attn (standalone, calibrated)", STACKER_NAME,
      "Cascade (v3-style, raw conf, fixed 0.85/0.15)",
      "Cascade v4 (calibrated P(malicious), tuned gate)"]
summary = res_df.loc[[m for m in key if m in res_df.index],
                     [c for c in ["Accuracy", "F1-weighted", "PR-AUC", "FPR"] if c in res_df.columns]]
print(summary.to_string())
print(f"\nCascade v4 routing: Tier-2 invoked on {ambig_v4.mean()*100:.1f}% of test traffic "
     f"(v3-style cascade: {ambig_v3naive.mean()*100:.1f}%)")
if DOMAIN_COL:
    print(f"Tier-0 cache would resolve {cache_hit.mean()*100:.1f}% of traffic instantly.")
print(f"\nOOD (Indonesian threats) -- {len(ood_df)} zero-shot URLs")
print(f"  Predicted malicious : {(pred_ood != benign_idx).sum()}")
print("=" * 70)